<a href="https://www.kaggle.com/code/ahmedelwekel/fashion-product-classifier?scriptVersionId=269159336" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications import InceptionResNetV2

import shutil


import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras import models,layers
from tensorflow.keras.callbacks import LearningRateScheduler

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder





# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Load the structured data
dataset_path = '/kaggle/input/fashion-product-images-small/'
styles_df = pd.read_csv(os.path.join(dataset_path, 'styles.csv'),on_bad_lines='skip')

# Check the first few rows of the dataframe
styles_df.head()
styles_df.info()


In [ ]:
#check for missing values 
styles_df.isna().sum()

In [ ]:
# Drop all rows with missing values
styles_df.dropna(inplace=True)

# Verify that all missing values have been removed
print("\nMissing values after dropping rows:")
print(styles_df.isnull().sum())

# Check the updated shape of the DataFrame
print("\nUpdated DataFrame shape:", styles_df.shape)


In [ ]:

# Define the path where your images are stored
image_dir = '/kaggle/input/fashion-product-images-small/images/'

# Create a new column with the complete image paths
styles_df['image_path'] = styles_df['id'].astype(str) + '.jpg'

# Check if the image file exists for each row
styles_df['image_exists'] = styles_df['image_path'].apply(lambda x: os.path.exists(os.path.join(image_dir, x)))

# Filter rows to keep only those with valid image paths
styles_df = styles_df[styles_df['image_exists']]

# Drop the helper column
styles_df.drop(columns=['image_exists'], inplace=True)

# Reset the index after filtering
styles_df.reset_index(drop=True, inplace=True)

# Check the updated DataFrame
print(f"Number of rows after dropping missing images: {styles_df.shape[0]}")



In [ ]:

styles_df.head()



In [ ]:

# Encode the target label
label_encoder = LabelEncoder()
styles_df['masterCategory'] = label_encoder.fit_transform(styles_df['masterCategory'])

# Store the label mapping if needed later for predictions
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(label_mapping)



In [ ]:

# Count the occurrences of each class
class_counts = styles_df['masterCategory'].value_counts()

# Keep only classes with more than 1 sample
styles_df = styles_df[styles_df['masterCategory'].isin(class_counts[class_counts > 1].index)]

# Now, split the dataset
train_df, val_df = train_test_split(styles_df, test_size=0.2, stratify=styles_df['masterCategory'])


# Split the dataset into training and validation sets (stratified to balance classes)
train_df, val_df = train_test_split(
    styles_df, 
    test_size=0.2, 
    stratify=styles_df['masterCategory'], 
    random_state=42
)

# Reset indices of train and validation DataFrames
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

print(f"Training samples: {len(train_df)}, Validation samples: {len(val_df)}")
print(train_df['masterCategory'].unique())

In [ ]:
# Shuffle the DataFrame
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

#Convert 'masterCategory' to string type explicitly
train_df['masterCategory'] = train_df['masterCategory'].astype(str)
val_df['masterCategory'] = val_df['masterCategory'].astype(str)


image_size = (224, 224)  # ResNet50 expects 224x224 images
batch_size = 32

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# For validation data, just rescaling
val_datagen = ImageDataGenerator(rescale=1./255)

# Flow images from dataframe
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/kaggle/input/fashion-product-images-small/images/',  # Adjust the path
    x_col='image_path',
    y_col='masterCategory',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='sparse',
    shuffle=False

)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory='/kaggle/input/fashion-product-images-small/images/',  # Adjust the path
    x_col='image_path',
    y_col='masterCategory',
    target_size=image_size,
    batch_size=batch_size,
    class_mode='sparse',
    shuffle=False
)




In [ ]:
# Load ResNet50 pre-trained on ImageNet, excluding the top classification layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model layers to prevent them from being updated during the initial training
base_model.trainable = False


In [ ]:
print( len(train_generator.class_indices))


In [ ]:




# Define the input shape (224x224x3 for standard image sizes)
input_shape = (224, 224, 3)

# Load the InceptionResNetV2 base model without the top layers (pre-trained on ImageNet)
base_model = InceptionResNetV2(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the base model initially to retain pre-trained weights
base_model.trainable = False

# Get the number of classes from the train generator
num_classes = len(train_generator.class_indices)

# Define the input layer
input_layer = layers.Input(shape=input_shape)

# Pass input through InceptionResNetV2 base model
x = base_model(input_layer)

# Add global average pooling to reduce the spatial dimensions
x = layers.GlobalAveragePooling2D()(x)

# Add a series of dense layers with dropout and batch normalization
x = layers.Dense(2048, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(1024, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

# Add an additional dense layer for complexity
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

# Output layer with softmax activation for multi-class classification
output_layer = layers.Dense(num_classes, activation='softmax')(x)

# Build the model
model = models.Model(inputs=input_layer, outputs=output_layer)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',  # Sparse for integer labels
              metrics=['accuracy'])

# Model summary to verify the architecture
model.summary()



In [ ]:


# Define the learning rate scheduler function
def scheduler(epoch, lr):
    if epoch > 5:  # Decrease the learning rate after epoch 10
        return lr * 0.1  # Decrease by 10%
    return lr  # Keep the learning rate as is for the first 10 epochs

# Define the learning rate scheduler callback
lr_scheduler = LearningRateScheduler(scheduler)

# Train the model for 15 epochs with learning rate scheduling
history = model.fit(
    train_generator,
    epochs=8,  # Number of epochs
    validation_data=val_generator,
    callbacks=[lr_scheduler]  # Apply learning rate scheduling
)

# Function to plot training & validation loss and accuracy
def plot_training_history(history):
    # Plot accuracy
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy over epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss over epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Plot the training history
plot_training_history(history)
